# CockroachDB Connector Unit Tests (v2.0)

**🆕 Version 2.0 - Catalog/Schema Support & Dual-Format CDC**

Test `LakeflowConnect` class with local credentials:
- **Direct Mode**: Sinkless changefeed (direct database connection)
- **Azure Parquet Mode**: Read from Azure Blob Storage Parquet files  
- **Azure JSON Mode**: Read from Azure Blob Storage JSON files (NEW!)
- **Dual Format Mode**: Both JSON + Parquet changefeeds (NEW!)
- **Volume Mode**: Read from Databricks Unity Catalog Volume

**Prerequisites:**
1. Save credentials in `../.env/` folder (see Cell 2 for loading)
2. Run this notebook in Databricks or local Jupyter with PySpark
3. For Volume Mode: Requires Databricks runtime with Unity Catalog access

**⚠️ Breaking Changes in v2.0:**
- REQUIRED: `catalog` parameter (database name)
- REQUIRED: `schema` parameter (schema name)
- OPTIONAL: `format` parameter ('parquet', 'json', or 'both')
- Paths are now auto-constructed: `{format}/{catalog}/{schema}/`

## Setup: Import and Load Credentials


In [ ]:
import sys
import json
import os
from pathlib import Path
import git

# Setup paths
repo = git.Repo(Path().absolute(), search_parent_directories=True)
git_root = repo.working_tree_dir
sys.path.insert(0, os.path.join(git_root, 'sources', 'cockroachdb'))
sys.path.insert(0, os.path.join(git_root, 'sources'))
sys.path.insert(0, git_root)

# Load credentials from JSON (preferred format for notebooks and Autoloader)
credentials_file = os.path.join(git_root, "sources/cockroachdb/examples/credentials_parquet.json")

# Fallback order: try parquet, then both, then json
credential_files = [
    os.path.join(git_root, "sources/cockroachdb/examples/credentials_parquet.json"),
    os.path.join(git_root, "sources/cockroachdb/examples/credentials_both.json"),
    os.path.join(git_root, "sources/cockroachdb/examples/credentials_json.json"),
    os.path.join(git_root, "sources/cockroachdb/.env/credentials.json"),  # Local override
]

credentials_config = None
for cred_file in credential_files:
    if os.path.exists(cred_file):
        with open(cred_file, 'r') as f:
            credentials_config = json.load(f)
        print(f"✅ Loaded credentials from: {os.path.basename(cred_file)}")
        break

if not credentials_config:
    print(f"⚠️  No credential files found. Checked:")
    for cf in credential_files:
        print(f"   - {cf}")
    print(f"\n💡 Create credentials file from template:")
    print(f"   cp sources/cockroachdb/examples/credentials_parquet.json sources/cockroachdb/.env/credentials.json")
    credentials_config = {}

# Import connector
from cockroachdb import LakeflowConnect

print(f"✅ Ready | Git root: {git_root}")


In [ ]:
# Display loaded configuration
print("="*80)
print("LOADED CONFIGURATION")
print("="*80)

if credentials_config:
    # Mask sensitive fields
    safe_config = {k: "***REDACTED***" if any(s in k.lower() for s in ['key', 'password', 'token']) 
                   else v for k, v in credentials_config.items()}
    print(json.dumps(safe_config, indent=2))
else:
    print("⚠️  No configuration loaded")
    
print("="*80)

In [ ]:
# Build test credentials from JSON config
credentials = {}

if credentials_config:
    # Direct mode: Use config as-is (already has catalog, schema, format)
    credentials["direct_mode"] = {**credentials_config}
    
    # Azure Parquet/JSON mode: Add Azure credentials if present
    if all(k in credentials_config for k in ["azure_storage_account", "azure_storage_key"]):
        credentials["azure_parquet_mode"] = {**credentials_config}
    
    # Volume mode: For testing only
    credentials["volume_mode"] = {
        "catalog": credentials_config.get("catalog", "main"),
        "schema": credentials_config.get("schema", "public"),
        "volume_path": "/Volumes/main/robert_lee_cockroachdb/parquet_files"
    }

print(f"✅ Credentials ready | Modes: {', '.join(credentials.keys())}")
if credentials.get("direct_mode"):
    print(f"   📁 Catalog: {credentials['direct_mode'].get('catalog', 'N/A')}")
    print(f"   📂 Schema: {credentials['direct_mode'].get('schema', 'N/A')}")
    print(f"   📊 Format: {credentials['direct_mode'].get('format', 'N/A')}")


## Test 1: Direct Mode (Sinkless Changefeed)

In [ ]:
print("="*80)
print("TEST 1: Direct Mode Initialization")
print("="*80)

# Get direct mode credentials
direct_options = credentials.get('direct_mode', {})

print(f"\n📋 Options:")
for key, value in direct_options.items():
    if 'password' in key.lower() or 'key' in key.lower():
        print(f"  {key}: ***REDACTED***")
    else:
        print(f"  {key}: {value}")

# Initialize connector in Direct mode
try:
    connector_direct = LakeflowConnect(direct_options)
    print(f"\n✅ Connector initialized successfully!")
    print(f"   Mode: {connector_direct.mode}")
    print(f"   Host: {connector_direct.host}")
    print(f"   Database: {connector_direct.database}")
    print(f"   Schema: {connector_direct.schema}")
except Exception as e:
    print(f"\n❌ Initialization failed: {e}")
    import traceback
    traceback.print_exc()
    connector_direct = None


In [ ]:
if connector_direct:
    print("="*80)
    print("TEST 1: Direct Mode - read_table()")
    print("="*80)
    
    table_name = "usertable"
    start_offset = {}  # First run
    table_options = {
        "initial_scan": "only",  # Snapshot only for testing
    }
    
    print(f"\n📖 Reading table: {table_name}")
    print(f"   Start offset: {start_offset}")
    print(f"   Options: {table_options}")
    
    try:
        rows_iter, end_offset = connector_direct.read_table(
            table_name, 
            start_offset, 
            table_options
        )
        
        # Collect first 10 rows for testing
        rows = []
        for idx, row in enumerate(rows_iter):
            if idx >= 10:  # Limit for testing
                break
            rows.append(row)
        
        print(f"\n✅ Successfully read {len(rows)} rows (limited to 10 for testing)")
        print(f"\n📊 End offset: {end_offset}")
        
        if rows:
            print(f"\n📋 First row:")
            print(json.dumps(rows[0], indent=2, default=str))
            
            print(f"\n📋 Row keys: {list(rows[0].keys())}")
        else:
            print("\n⚠️  No rows returned (table may be empty)")
            
    except Exception as e:
        print(f"\n❌ read_table() failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭️  Skipping read_table() test (connector not initialized)")


## Test 2: Azure Parquet Mode

- setup
source/setup_azure_blob_for_cdc.sh setup storage 
- create changefood

sources/cockroachdb_s3/scripts/test_azure_cdc.sh

In [ ]:
print("="*80)
print("TEST 2: Azure Parquet Mode Initialization")
print("="*80)

# Get Azure Parquet mode credentials
azure_options = credentials.get('azure_parquet_mode', {})

print(f"\n📋 Options:")
for key, value in azure_options.items():
    if 'password' in key.lower() or 'key' in key.lower():
        print(f"  {key}: ***REDACTED***")
    else:
        print(f"  {key}: {value}")

# Initialize connector in Azure Parquet mode
try:
    connector_azure = LakeflowConnect(azure_options)
    print(f"\n✅ Connector initialized successfully!")
    print(f"   Mode: {connector_azure.mode}")
    print(f"   Host: {connector_azure.host}")
    print(f"   Azure Account: {connector_azure.azure_account_name}")
    print(f"   Container: {connector_azure.azure_container}")
    print(f"   Path Prefix: {connector_azure.azure_path_prefix}")
except Exception as e:
    print(f"\n❌ Initialization failed: {e}")
    import traceback
    traceback.print_exc()
    connector_azure = None


In [ ]:
if connector_azure:
    print("="*80)
    print("TEST 2: Azure Parquet Mode - read_table()")
    print("="*80)
    print("\n⚠️  Note: This test requires network connectivity to Azure Blob Storage")
    print("   Designed for: Databricks environment")
    print("   Local testing: May fail with DNS/network errors")
    print("")
    
    # Check if we're likely running locally (no network to Azure)
    import socket
    azure_host = f"{connector_azure.azure_account_name}.blob.core.windows.net"
    
    try:
        # Quick DNS check
        socket.gethostbyname(azure_host)
        network_available = True
        print(f"✅ Network check: Can resolve {azure_host}")
    except socket.gaierror:
        network_available = False
        print(f"❌ Network check: Cannot resolve {azure_host}")
        print("   This is expected when running locally without Azure access")
    
    if network_available:
        table_name = "usertable"
        start_offset = {}
        table_options = {"initial_scan": "yes"}
        
        print(f"\n📖 Reading table: {table_name}")
        
        try:
            rows_iter, end_offset = connector_azure.read_table(
                table_name, 
                start_offset, 
                table_options
            )
            
            # Collect first 10 rows
            rows = []
            for idx, row in enumerate(rows_iter):
                if idx >= 10:
                    break
                rows.append(row)
            
            print(f"\n✅ Successfully read {len(rows)} rows")
            print(f"📊 End offset: {end_offset}")
            
            if rows:
                print(f"\n📋 First row:")
                print(json.dumps(rows[0], indent=2, default=str))
            
            else:
                print("\n⚠️  No rows returned")
                    
        except Exception as e:
            print(f"\n❌ read_table() failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("\n⏭️  Skipping Azure Parquet test (no network connectivity)")
        print("   To test this mode:")
        print("   1. Run this notebook in Databricks")
        print("   2. Or ensure local network access to Azure")
else:
    print("⏭️  Skipping test (connector not initialized)")


## Test 3: Volume Mode (Databricks Only)


In [ ]:
print("="*80)
print("TEST 3: Volume Mode Initialization")
print("="*80)

# Get Volume mode credentials
volume_options = credentials.get('volume_mode', {})

print(f"\n📋 Options:")
for key, value in volume_options.items():
    print(f"  {key}: {value}")

# Initialize connector in Volume mode
try:
    connector_volume = LakeflowConnect(volume_options)
    print(f"\n✅ Connector initialized successfully!")
    print(f"   Mode: {connector_volume.mode}")
    print(f"   Volume Path: {connector_volume.volume_path}")
    print(f"   Schema: {connector_volume.schema}")
except Exception as e:
    print(f"\n❌ Initialization failed: {e}")
    import traceback
    traceback.print_exc()
    connector_volume = None


In [ ]:
if connector_volume:
    print("="*80)
    print("TEST 3: Volume Mode - read_table()")
    print("="*80)
    print("⚠️  This test requires Databricks runtime with access to Volumes")
    
    table_name = "usertable"
    start_offset = {}  # First run
    table_options = {}
    
    print(f"\n📖 Reading table: {table_name}")
    print(f"   Start offset: {start_offset}")
    
    try:
        rows_iter, end_offset = connector_volume.read_table(
            table_name, 
            start_offset, 
            table_options
        )
        
        # Collect first 10 rows for testing
        rows = []
        for idx, row in enumerate(rows_iter):
            if idx >= 10:  # Limit for testing
                break
            rows.append(row)
        
        print(f"\n✅ Successfully read {len(rows)} rows (limited to 10 for testing)")
        print(f"\n📊 End offset: {end_offset}")
        
        if rows:
            print(f"\n📋 First row:")
            print(json.dumps(rows[0], indent=2, default=str))
            
            print(f"\n📋 Row keys: {list(rows[0].keys())}")
            
            # Show CDC metadata
            if '_cdc_key' in rows[0]:
                print(f"\n📊 CDC Metadata:")
                print(f"   _cdc_key: {rows[0].get('_cdc_key')}")
                print(f"   _cdc_updated: {rows[0].get('_cdc_updated')}")
                print(f"   _cdc_operation: {rows[0].get('_cdc_operation')}")
                print(f"   _source_file: {rows[0].get('_source_file', 'N/A')[:50]}...")
        else:
            print("\n⚠️  No rows returned (Volume may be empty)")
            
    except Exception as e:
        print(f"\n❌ read_table() failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭️  Skipping read_table() test (connector not initialized)")


## Test Summary


In [ ]:
print("="*80)
print("TEST SUMMARY")
print("="*80)

results = [
    ("Direct Mode", connector_direct is not None),
    ("Azure Parquet Mode", connector_azure is not None),
    ("Volume Mode", connector_volume is not None),
]

for mode, success in results:
    status = "✅ PASS" if success else "❌ FAIL"
    print(f"{status} - {mode}")

print("="*80)
print("\n💡 Next Steps:")
print("   - Check output above for any errors")
print("   - Verify row data matches expectations")
print("   - Test with different table_options (initial_scan, cursor, etc.)")
print("   - Test cursor resumption (use end_offset as start_offset)")
